# Analise arquivo raw produtos.csv

In [33]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [34]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR =  os.path.join(DATA_DIR, 'raw')

In [35]:
df = pd.read_csv(os.path.join(RAW_DIR, 'produtos.csv'), sep=';')
df.shape

(81, 6)

## Analise exploratória

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   81 non-null     str    
 1   descricao    80 non-null     str    
 2   categoria    80 non-null     str    
 3   preco_custo  80 non-null     float64
 4   preco_venda  80 non-null     str    
 5   unidade      80 non-null     str    
dtypes: float64(1), str(5)
memory usage: 3.9 KB


In [37]:
df.isna().sum()

id_produto     0
descricao      1
categoria      1
preco_custo    1
preco_venda    1
unidade        1
dtype: int64

In [38]:
df.head()

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
0,P0001,Água Econômico,Bebidas,140.86,212.54,L
1,P0002,Saco plástico Premium,Embalagens,155.90,234.36,CX
2,P0003,Sabonete Premium,Higiene,133.11,81.73,L
3,P0004,Suco Premium,Bebidas,142.15,28.62,UN
4,P0005,Shampoo Premium,Higiene,147.35,112.09,KG


In [39]:
df.tail()

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
76,P0077,Filme plástico 5kg,Embalagens,16.32,12.45,L
77,P0078,Desinfetante Tradicional,Limpeza,18.26,230.97,UN
78,P0079,Detergente 12un,Limpeza,136.62,114.51,L
79,P0080,Sabão 500ml,Limpeza,79.59,195.98,UN
80,P0063,Bebida energética Tradicional,Bebidas,157.34,139.05,L


In [40]:
df.sample(5)

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
22,P0023,Refrigerante 12un,Bebidas,172.52,125.76,UN
1,P0002,Saco plástico Premium,Embalagens,155.90,234.36,CX
15,P0016,Bebida energética 12un,Bebidas,105.24,"89,90",L
59,P0060,Refrigerante 1L,Bebidas,88.19,174.73,KG
39,P0040,Feijão Premium,Alimentos,151.38,150.26,CX


In [41]:
df.isnull().sum()

id_produto     0
descricao      1
categoria      1
preco_custo    1
preco_venda    1
unidade        1
dtype: int64

In [42]:
q = '''SELECT id_produto
            , descricao
            , categoria
            , preco_custo
            , preco_venda
            , unidade
        FROM df
        WHERE descricao isnull
                    or categoria isnull
                    or preco_custo isnull
                    or preco_venda isnull
                    or unidade isnull
'''

In [43]:
print(pysqldf(q))

  id_produto                   descricao   categoria  preco_custo preco_venda  \
0      P0028  Água sanitária Tradicional     Limpeza       129.01       58.57   
1      P0032          Filme plástico 5kg  Embalagens         3.05         NaN   
2      P0045                   Café 12un   Alimentos          NaN      165.17   
3      P0058                         NaN     Limpeza        56.58       57.64   
4      P0076                Sabonete 5kg         NaN       124.58      168.84   

  unidade  
0     NaN  
1       L  
2      UN  
3      CX  
4      KG  


In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   81 non-null     str    
 1   descricao    80 non-null     str    
 2   categoria    80 non-null     str    
 3   preco_custo  80 non-null     float64
 4   preco_venda  80 non-null     str    
 5   unidade      80 non-null     str    
dtypes: float64(1), str(5)
memory usage: 3.9 KB


## Tratamento de dados

In [45]:
df_original = df.copy()

In [46]:
df = df.fillna('NAO INFORMADO')

In [47]:
df[['preco_custo', 'preco_venda']] = \
df[['preco_custo', 'preco_venda']].replace('NAO INFORMADO', 0)

In [48]:
df['preco_custo'] = pd.to_numeric(df['preco_custo'], errors='coerce')

In [49]:
df['preco_venda'] = pd.to_numeric(df['preco_venda'], errors='coerce')

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   81 non-null     str    
 1   descricao    81 non-null     str    
 2   categoria    81 non-null     str    
 3   preco_custo  81 non-null     float64
 4   preco_venda  80 non-null     float64
 5   unidade      81 non-null     str    
dtypes: float64(2), str(4)
memory usage: 3.9 KB


In [51]:
df.sample(10)

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
49,P0050,Creme dental 12un,Higiene,132.12,66.61,CX
17,P0018,Macarrão 500ml,Alimentos,120.17,149.76,UN
6,P0007,Macarrão Econômico,Alimentos,116.18,97.72,KG
32,P0033,Shampoo 5kg,Higiene,41.06,106.29,UN
25,P0026,Filme plástico Premium,Embalagens,134.37,115.50,KG
42,P0043,Suco Tradicional,Bebidas,105.01,223.15,L
1,P0002,Saco plástico Premium,Embalagens,155.90,234.36,CX
5,P0006,Refrigerante Tradicional,Bebidas,119.49,97.44,KG
28,P0029,Detergente Premium,Limpeza,28.85,93.78,CX
67,P0068,Filme plástico 5kg,Embalagens,163.71,73.47,L


## Salvando dados de produtos em Bronze

In [52]:
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')

In [53]:
df.to_csv(os.path.join(BRONZE_DIR, 'b_produtos.csv'), index=False)